<a href="https://colab.research.google.com/github/simon-clematide/colab-notebooks-for-teaching/blob/main/plt1_real_world_data_xml_json_spacy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Real-World Data: XML, spaCy, JSON

Companion to *Programming in Language Technology 1*, Session 11, "Real-World Data and NLP Tooling" (University of Zurich).

**Question:** Who and what is in the SAC yearbook of 1940?
We follow one pipeline: XML file → sentences → records → JSONL file → counts.

How to work: read the claim, write down your **prediction**, then run the cell. Answers to *Check yourself* are collapsed.
Sections marked *(optional)* can be skipped. Everything runs on a free Colab CPU in seconds.

## N0 Setup

Downloads an excerpt of the yearbook (2 articles, 134 sentences), two result files for the *whole* yearbook (14,485 sentences), and this notebook's own file. About 3 MB.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve

BASE = "https://raw.githubusercontent.com/simon-clematide/colab-notebooks-for-teaching/main/"
FILES = ["data/sac_1940_excerpt.xml", "data/entities_spacy_xx.jsonl",
         "data/entities_hf.jsonl", "plt1_real_world_data_xml_json_spacy.ipynb"]
for name in FILES:
    path = Path(name)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        urlretrieve(BASE + name, path)
    print(f"{path.stat().st_size:>9,} bytes  {path}")

  158,875 bytes  data/sac_1940_excerpt.xml
1,490,852 bytes  data/entities_spacy_xx.jsonl
1,283,327 bytes  data/entities_hf.jsonl
   17,993 bytes  plt1_real_world_data_xml_json_spacy.ipynb


## N1 A notebook is JSON, too

*Slide: "A Notebook Is JSON, Too".*

**Claim:** A notebook file (`.ipynb`) is plain JSON: a dict whose `cells` are a list of dicts.
Colab, Jupyter, and VS Code only *render* it.

**Predict:** Which keys does one cell have? What type is its `source`?

In [2]:
import json

with open("plt1_real_world_data_xml_json_spacy.ipynb", encoding="utf-8") as f:
    notebook = json.load(f)

print(type(notebook).__name__, list(notebook))
cell = notebook["cells"][2]
print(list(cell))
print(type(cell["source"]).__name__, cell["source"][:2])

dict ['cells', 'metadata', 'nbformat', 'nbformat_minor']
['cell_type', 'execution_count', 'id', 'metadata', 'outputs', 'source']
list ['from pathlib import Path\n', 'from urllib.request import urlretrieve\n']


In [3]:
from collections import Counter

Counter(cell["cell_type"] for cell in notebook["cells"])

Counter({'markdown': 23, 'code': 20})

**Check yourself.**

1. Why is `source` a list of strings and not one string?
2. Where are the outputs of a code cell stored?

<details>
<summary>Answers: list of lines; `outputs`</summary>

1. It stores the cell's text line by line; `"".join(cell["source"])` gives the text. A format decision, like our records.
2. In the cell's `outputs` list. That is why a notebook downloaded from GitHub can show results before you run anything.

</details>

## N2 Reading the tree: children or descendants?

*Slides: "Parsing a File into Element Objects", "Children or Descendants?", "Stage 1: A Generator of Sentences".*

Sentence `43-6` contains a picture caption, which is a sentence `<s>` of its own *inside* it.

**Predict:** `findall("w")` visits direct children, `iter("w")` all descendants. How many words does each count for sentence 43-6?

In [4]:
import xml.etree.ElementTree as ET

book = ET.parse("data/sac_1940_excerpt.xml").getroot()
s = next(s for s in book.iter("s") if s.get("n") == "43-6")
print(len(s.findall("w")), "children,", len(list(s.iter("w"))), "descendants")
print(" ".join(w.text for w in s.findall("w")))

22 children, 129 descendants
Der Genfer Saussure ist , mehr als jeder andere vor ihm , der Entdecker und Deuter der Landschaft des Hochgebirges geworden .


A recursive function prints the tree structure of the sentence (from the PCL1 notebook *XML Processing Basics*):

In [5]:
def print_tags(element, indent="", depth=3):
    print(indent + element.tag, dict(element.attrib))
    if depth > 0:
        for child in element[:3]:
            print_tags(child, indent + "    ", depth - 1)

print_tags(s)

s {'lang': 'de', 'n': '43-6'}
    w {'lemma': 'd', 'n': '43-6-1', 'pos': 'ART'}
    w {'lemma': 'Genfer', 'n': '43-6-2', 'pos': 'ADJA'}
    w {'lemma': 'Saussure', 'n': '43-6-3', 'pos': 'NE'}


Stage 1 of our pipeline: a generator of `(n, lang, words)`; each `<s>` is a sentence, its words are its direct `<w>` children.

In [6]:
def read_sentences(path):
    book = ET.parse(path).getroot()
    for s in book.iter("s"):
        words = [w.text for w in s.findall("w") if w.text]
        if words:
            yield s.get("n"), s.get("lang"), words

sentences = {n: (lang, words) for n, lang, words in read_sentences("data/sac_1940_excerpt.xml")}
print(len(sentences), "sentences,", sum(len(w) for _, w in sentences.values()), "words")

134 sentences, 2661 words


**Check yourself.**

1. Why is `iter("w")` on every sentence wrong for counting the words of the book?
2. Is `iter` always wrong?

<details>
<summary>Answers: double counting; no</summary>

1. The caption's words are descendants of both sentences, so they are counted twice: 2,768 instead of 2,661 words in the excerpt.
2. `book.iter("s")` is right: we *want* every sentence, also nested ones. The decision depends on the question.

</details>

## N3 Annotating with spaCy

*Slides: "A Trained Pipeline in Two Lines" to "Entities Are Spans".*

Sentence `43-6` again, now annotated. We pass the words from the XML file, so spaCy does not tokenize again.
The German model downloads once (about 15 MB).

In [7]:
import sys
!{sys.executable} -m spacy download de_core_news_sm -q

✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')


In [8]:
import spacy
from spacy import displacy
from spacy.tokens import Doc

nlp_de = spacy.load("de_core_news_sm")
lang, words = sentences["43-6"]
doc = nlp_de(Doc(nlp_de.vocab, words=words))
for token in doc:
    print(f"{token.i:2} {token.text:10} {token.pos_:6} {token.lemma_:10} {token.dep_:6} -> {token.head.i}")

 0 Der        DET    der        nk     -> 2
 1 Genfer     ADJ    Genfer     nk     -> 2
 2 Saussure   NOUN   Saussur    sb     -> 3
 3 ist        AUX    sein       ROOT   -> 3
 4 ,          PUNCT  --         punct  -> 3
 5 mehr       PRON   mehr       pd     -> 20
 6 als        SCONJ  als        cm     -> 8
 7 jeder      DET    jeder      nk     -> 8
 8 andere     PRON   anderer    cc     -> 5
 9 vor        ADP    vor        mnr    -> 8
10 ihm        PRON   ihm        nk     -> 9
11 ,          PUNCT  --         punct  -> 8
12 der        DET    der        nk     -> 13
13 Entdecker  NOUN   Entdecker  app    -> 8
14 und        CCONJ  und        cd     -> 13
15 Deuter     NOUN   Deuter     cj     -> 14
16 der        DET    der        nk     -> 17
17 Landschaft NOUN   Landschaft ag     -> 13
18 des        DET    der        nk     -> 19
19 Hochgebirges NOUN   Hochgebirg ag     -> 17
20 geworden   AUX    werden     sb     -> 3
21 .          PUNCT  --         punct  -> 3


**Predict:** You know dependency parses. Starting from the token `Saussure` in Python, how do you get the **lemma of its governor**?

In [9]:
saussure = doc[2]
saussure.head.lemma_

'sein'

In [10]:
displacy.render(doc, style="dep", jupyter=True, options={"compact": True, "distance": 90})
displacy.render(doc, style="ent", jupyter=True)

spaCy has its own JSON export. **Predict:** do its entity `start`/`end` count tokens or characters?

In [11]:
doc.to_json()["ents"]

[{'start': 4, 'end': 19, 'label': 'MISC'},
 {'start': 102, 'end': 114, 'label': 'MISC'}]

**Check yourself.**

1. What type is `token.head`?
2. Look at the lemma of *Saussure* and the label of *Genfer Saussure*. What do they tell you?

<details>
<summary>Answers: a `Token`; model guesses</summary>

1. A `Token` object, so `token.head.lemma_` works: an attribute of an attribute, as in L10.
2. Lemmas and labels are predictions. A model makes mistakes even in its own language; check before you count.

</details>

## N4 Models have a language

*Slides: "Entities in the Whole Yearbook", "A Multilingual Model in the Same Slot".*

The same French sentence (`58-21`), two models. The multilingual model downloads once (about 10 MB).

**Predict:** Which place is in the sentence? Which model finds it?

In [12]:
import sys
!{sys.executable} -m spacy download xx_ent_wiki_sm -q

✔ Download and installation successful
You can now load the package via spacy.load('xx_ent_wiki_sm')


In [13]:
nlp_xx = spacy.load("xx_ent_wiki_sm")
lang, words = sentences["58-21"]
print(lang, " ".join(words))
for nlp in (nlp_de, nlp_xx):
    doc = nlp(Doc(nlp.vocab, words=words))
    print(f"{nlp.meta['lang']}_{nlp.meta['name']:16}", [(e.text, e.label_) for e in doc.ents])

fr Je partais pour le Lac Noir , tout fier de mon équipement neuf :
de_core_news_sm     [('fier de mon équipement neuf', 'PER')]
xx_ent_wiki_sm      [('Lac Noir', 'LOC')]


**Predict:** Is the multilingual model now right on French? Try sentence `58-51`.

In [14]:
lang, words = sentences["58-51"]
print(" ".join(words))
doc = nlp_xx(Doc(nlp_xx.vocab, words=words))
[(e.text, e.label_) for e in doc.ents]

marcher non seulement avec leurs jambes , mais surtout avec leur intelligence ;


[('marcher', 'MISC')]

**Check yourself.**

1. What did the swap fix in 58-21, and what can one sentence not show?
2. Name the three kinds of result you must distinguish when judging a model.

<details>
<summary>Answers: one sentence; three cases</summary>

1. The German model misses *Lac Noir* (and may report a false alarm); the multilingual one finds it. Whether the model is better on French in general needs a sample of sentences judged by a person.
2. A false alarm (found, but not an entity), a miss (an entity not found), and an error shared by both models.

</details>

## N5 Records in JSON and JSONL

*Slides: "JSON: Text for Nested Data" to "The Answer".*

**Predict:** What does `json.dumps` make of `Zürich`, and of a tuple?

In [15]:
record = {"n": "58-21", "lang": "fr", "entities": [{"start": 4, "end": 6, "label": "LOC", "text": "Lac Noir"}]}
print(json.dumps({"place": "Zürich", "span": (4, 6)}))
print(json.dumps({"place": "Zürich"}, ensure_ascii=False))
json.loads(json.dumps(record)) == record

{"place": "Z\u00fcrich", "span": [4, 6]}
{"place": "Zürich"}


True

The result file of the multilingual model for the **whole** yearbook: one record per line. We read it lazily.

In [16]:
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)

records = list(read_jsonl("data/entities_spacy_xx.jsonl"))
print(len(records), "records")
records[3]

14485 records


{'n': '0-4',
 'lang': 'de',
 'entities': [{'start': 1,
   'end': 4,
   'label': 'PER',
   'text': 'G. O. Dyhrenfurth'}]}

**Use the record:** an entity says only `start` and `end`. Recover the mention from the words of its sentence.
**Predict:** what does `words[start:end]` give for the first entity of sentence `58-21`?

In [17]:
by_n = {record["n"]: record for record in records}
lang, words = sentences["58-21"]
entity = by_n["58-21"]["entities"][0]
print(entity)
words[entity["start"]:entity["end"]]

{'start': 4, 'end': 6, 'label': 'LOC', 'text': 'Lac Noir'}


['Lac', 'Noir']

**The answer** to our question: count predicted mentions per label.

In [18]:
counts = {}
for record in records:
    for entity in record["entities"]:
        counts.setdefault(entity["label"], Counter())[entity["text"]] += 1
for label in ("PER", "LOC"):
    print(label, counts[label].most_common(6))

PER [('Burgener', 70), ('Alexander', 54), ('Whymper', 41), ('Hans', 25), ('Coolidge', 23), ('Saussure', 23)]
LOC [('Alpes', 130), ('Cervin', 65), ('Zermatt', 61), ('Mont Blanc', 58), ('Alpen', 56), ('Nebel', 54)]


**Check yourself.**

1. Can we add the 70 *Burgener* and the 54 *Alexander* and report 124 mentions of Alexander Burgener?
2. The JSONL file lets us recount names without spaCy. What can we *not* do with it alone?

<details>
<summary>Answers: no; judge in context</summary>

1. The counter groups predicted mentions by label and text. Whether two mentions refer to the same person needs evidence from the sentences; *Alexander* may be someone else.
2. Judge whether a label fits: that needs the source sentence, which we find again through `n` in the XML file.

</details>

## N6 Two models, one yearbook *(optional)*

*Slide: "Two Models, One Yearbook".* No model needed: we compare two result files.

In [19]:
def spans(path):
    return {(r["n"], e["start"], e["end"]): e["label"]
            for r in read_jsonl(path) for e in r["entities"]}

a = spans("data/entities_spacy_xx.jsonl")
b = spans("data/entities_hf.jsonl")
both = a.keys() & b.keys()
print(len(a), len(b), len(both), sum(a[k] == b[k] for k in both))

13459 9982 6976 5659


**Check yourself.**

1. What do 6,976 shared spans say about accuracy?

<details>
<summary>Answers: nothing alone</summary>

1. They measure agreement. Both models can agree on an error, and a place that neither model found does not appear at all.

</details>

## N7 Your turn *(optional, after class)*

Write `read_articles(path)`: one record **per article** (`<article n="...">`) with its number, the numbers of its first and last sentence, and its word count.
Write the records as JSONL to `articles.jsonl` and read the file back.
(Idea from the PCL1 notebook *XML to JSONL processing* on the British Alpine Club journal.)

In [20]:
# your code here